# My Exercise Solutions: Chapter 3 (Attention Mechanisms)

**Date**: February 21, 2026

**My goal**: Work through the ch03 exercises in my own words and code to solidify my understanding of self-attention and multi-head attention.

**Zero-copy note**: This is my personal synthesis. I am not copying the source-material solutions — I'm reconstructing the logic myself after reading the chapter.

**Attribution**: Concepts based on *Build a Large Language Model From Scratch* by Sebastian Raschka.

In [ ]:
from importlib.metadata import version
import torch
import torch.nn as nn

torch.manual_seed(123)
print("torch version:", version("torch"))

## Shared Setup

The same 6-token input sequence used throughout ch03. Each token is a 3D embedding vector.

In [ ]:
# 6-token input: "Your journey starts with one step"
# Each row is a 3D token embedding
inputs = torch.tensor(
    [[0.43, 0.15, 0.89],  # "Your"     x^(1)
     [0.55, 0.87, 0.66],  # "journey"  x^(2)
     [0.57, 0.85, 0.64],  # "starts"   x^(3)
     [0.22, 0.58, 0.33],  # "with"     x^(4)
     [0.77, 0.25, 0.10],  # "one"      x^(5)
     [0.05, 0.80, 0.55]]  # "step"     x^(6)
)

d_in, d_out = 3, 2
print(f"inputs shape: {inputs.shape}  (6 tokens x 3 dims)")

## Exercise 3.1

**Prompt (my words)**: `SelfAttention_v1` uses `nn.Parameter` with raw weight matrices, while `SelfAttention_v2` uses `nn.Linear` (no bias). They should be equivalent — show that by transferring v2's weights into v1 and verifying the outputs match.

**My approach**: The key insight is that `nn.Linear` stores weights **transposed** relative to `nn.Parameter`. So `W_query` in v1 has shape `(d_in, d_out)`, but `nn.Linear.weight` has shape `(d_out, d_in)`. I need to `.T` when copying.

In [ ]:
class SelfAttention_v1(nn.Module):
    """Attention using raw nn.Parameter weight matrices."""

    def __init__(self, d_in: int, d_out: int) -> None:
        super().__init__()
        # Shape: (d_in, d_out) — raw parameter, no bias
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        queries = x @ self.W_query
        keys    = x @ self.W_key
        values  = x @ self.W_value

        # Scaled dot-product attention
        attn_scores   = queries @ keys.T
        attn_weights  = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        return attn_weights @ values


class SelfAttention_v2(nn.Module):
    """Attention using nn.Linear (better weight init, no bias)."""

    def __init__(self, d_in: int, d_out: int, qkv_bias: bool = False) -> None:
        super().__init__()
        # nn.Linear stores weight as (d_out, d_in) internally
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        queries = self.W_query(x)
        keys    = self.W_key(x)
        values  = self.W_value(x)

        attn_scores  = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        return attn_weights @ values


torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)

torch.manual_seed(123)
sa_v2 = SelfAttention_v2(d_in, d_out)

In [ ]:
# Transfer v2 weights into v1.
# nn.Linear.weight shape: (d_out, d_in) — needs .T to match v1's (d_in, d_out)
sa_v1.W_query = nn.Parameter(sa_v2.W_query.weight.T)
sa_v1.W_key   = nn.Parameter(sa_v2.W_key.weight.T)
sa_v1.W_value = nn.Parameter(sa_v2.W_value.weight.T)

out_v1 = sa_v1(inputs)
out_v2 = sa_v2(inputs)

print("v1 output:\n", out_v1)
print("\nv2 output:\n", out_v2)
print("\nOutputs match:", torch.allclose(out_v1, out_v2, atol=1e-6))

**My takeaway**: The `.T` is the key. `nn.Linear` stores weights transposed for efficient matrix multiplication. Once I account for that, both classes produce identical results. On top of that, `nn.Linear` has smarter default weight initialization (Kaiming uniform), which is why it's preferred in practice.

## Exercise 3.2

**Prompt (my words)**: If I use `MultiHeadAttentionWrapper` and want the final output to have 2 dimensions (matching the single-head `d_out=2` case), what should `d_out` be when using 2 heads?

**My approach**: `MultiHeadAttentionWrapper` concatenates head outputs, so total output dim = `d_out * num_heads`. To get a final dim of 2 with 2 heads, I need `d_out = 1` per head.

In [ ]:
class CausalAttention(nn.Module):
    """Single-head causal (masked) self-attention."""

    def __init__(self, d_in: int, d_out: int, context_length: int,
                 dropout: float, qkv_bias: bool = False) -> None:
        super().__init__()
        self.d_out    = d_out
        self.W_query  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key    = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout  = nn.Dropout(dropout)
        # Causal mask: upper-triangular ones (above diagonal)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, n_tokens, _ = x.shape
        queries = self.W_query(x)
        keys    = self.W_key(x)
        values  = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        # Mask future positions with -inf so softmax drives them to 0
        attn_scores.masked_fill_(
            self.mask.bool()[:n_tokens, :n_tokens], -torch.inf
        )
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        return attn_weights @ values


class MultiHeadAttentionWrapper(nn.Module):
    """Multiple CausalAttention heads concatenated along the embedding dim."""

    def __init__(self, d_in: int, d_out: int, context_length: int,
                 dropout: float, num_heads: int, qkv_bias: bool = False) -> None:
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)]
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Each head outputs (b, n_tokens, d_out) → cat along last dim → (b, n_tokens, d_out*num_heads)
        return torch.cat([head(x) for head in self.heads], dim=-1)


# Batch: duplicate the 6-token input to simulate a batch of 2
batch = torch.stack((inputs, inputs), dim=0)  # (2, 6, 3)
context_length = batch.shape[1]

torch.manual_seed(123)
# d_out=1 per head × 2 heads = 2D output (matches single-head d_out=2 case)
mha_wrapper = MultiHeadAttentionWrapper(
    d_in=3, d_out=1, context_length=context_length, dropout=0.0, num_heads=2
)
context_vecs = mha_wrapper(batch)

print("context_vecs shape:", context_vecs.shape)  # expect (2, 6, 2)
print(context_vecs)

**My takeaway**: `MultiHeadAttentionWrapper` concatenates, so output dim = `d_out * num_heads`. The efficient `MultiHeadAttention` class (Section 3.6.2) is smarter — it lets me control the final output dim directly via `d_out`, regardless of head count.

## Exercise 3.3

**Prompt (my words)**: Instantiate a `MultiHeadAttention` module with GPT-2 small settings: context length 1024, embedding dim 768, 12 heads. Count the trainable parameters.

**My approach**: GPT-2 small has `d_model=768`, `num_heads=12`. The MHA module has `W_query`, `W_key`, `W_value` (each `768×768`), plus an output projection `out_proj` (`768×768`). That's 4 weight matrices. I'll count with `sum(p.numel() for p in model.parameters())` and reason through the math.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Efficient multi-head attention using weight splits (not head concatenation)."""

    def __init__(self, d_in: int, d_out: int, context_length: int,
                 dropout: float, num_heads: int, qkv_bias: bool = False) -> None:
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.d_out     = d_out
        self.num_heads = num_heads
        self.head_dim  = d_out // num_heads  # each head sees this many dims

        self.W_query  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key    = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # combines head outputs
        self.dropout  = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, n_tokens, d_in = x.shape

        # Project all heads at once, then reshape to split across heads
        keys    = self.W_key(x).view(b, n_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = self.W_query(x).view(b, n_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values  = self.W_value(x).view(b, n_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool   = self.mask.bool()[:n_tokens, :n_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Merge heads back: (b, n_tokens, d_out)
        context_vec = (attn_weights @ values).transpose(1, 2).contiguous()
        context_vec = context_vec.view(b, n_tokens, self.d_out)
        return self.out_proj(context_vec)


# GPT-2 small settings
context_length = 1024
d_in = d_out = 768
num_heads = 12

mha_gpt2 = MultiHeadAttention(d_in, d_out, context_length, dropout=0.0, num_heads=num_heads)

total_params = sum(p.numel() for p in mha_gpt2.parameters() if p.requires_grad)
print(f"Trainable parameters in MHA (GPT-2 small scale): {total_params:,}")

**My manual verification**:

- `W_query`: $768 \times 768 = 589{,}824$ + bias $768 = 590{,}592$
- `W_key`:   same = $590{,}592$
- `W_value`: same = $590{,}592$
- `out_proj`: same = $590{,}592$
- **Total**: $590{,}592 \times 4 = 2{,}362{,}368$

**My takeaway**: The MHA module in GPT-2 small has about 2.36M parameters. GPT-2 small has 117M total — so MHA is only a fraction of the full model. Most parameters are in the feed-forward layers and the token embedding table.